# 📦 1A — Thu thập Dataset
**Dự án:** Nhận diện Món ăn Việt Nam bằng AI  
**Mục tiêu notebook này:**
- Tải VietFood67 (33k ảnh, 68 class, có bbox) từ Kaggle
- Tải 30VNFoods (25k ảnh, 30 class) từ Kaggle
- Lưu vào Google Drive theo cấu trúc chuẩn
- Kiểm tra cấu trúc thư mục & báo cáo nhanh

**⚠️ Yêu cầu trước khi chạy:**
1. Có tài khoản Kaggle
2. Đã tạo Google Drive shared folder `VietFood-Project/`
3. Đã thêm Kaggle API token vào Colab Secrets (xem hướng dẫn Cell 2)

## 🔑 Bước 0 — Hướng dẫn lấy Kaggle API Token

1. Vào **kaggle.com** → Account → **Create New Token** → tải file `kaggle.json`
2. Mở file `kaggle.json`, bạn sẽ thấy dạng: `{"username": "...", "key": "..."}`
3. Trên Colab: click biểu tượng **🔑 (Secrets)** ở thanh trái
4. Thêm 2 secret:
   - `KAGGLE_USERNAME` = username của bạn
   - `KAGGLE_KEY` = key của bạn
5. Bật toggle **"Notebook access"** cho cả 2

In [ ]:
# ============================================================
# CELL 1: Mount Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive đã được mount!')

In [ ]:
# ============================================================
# CELL 2: Cấu hình Kaggle API từ Colab Secrets
# ============================================================
import os
from google.colab import userdata

# Lấy credentials từ Colab Secrets (an toàn, không hardcode)
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')

# Tạo file kaggle.json để kaggle CLI nhận diện
os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    import json
    json.dump({
        'username': os.environ['KAGGLE_USERNAME'],
        'key': os.environ['KAGGLE_KEY']
    }, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print('✅ Kaggle API đã được cấu hình!')

In [ ]:
# ============================================================
# CELL 3: Tạo cấu trúc thư mục trên Google Drive
# ============================================================
import os

# ⚙️ Chỉnh đường dẫn này nếu tên folder Drive của bạn khác
DRIVE_ROOT = '/content/drive/MyDrive/VietFood-Project'

DIRS = [
    f'{DRIVE_ROOT}/data/raw/VietFood67',
    f'{DRIVE_ROOT}/data/raw/30VNFoods',
    f'{DRIVE_ROOT}/data/processed',
    f'{DRIVE_ROOT}/data/crawled',
    f'{DRIVE_ROOT}/data/rejected',
    f'{DRIVE_ROOT}/reports',
    f'{DRIVE_ROOT}/labels',
    f'{DRIVE_ROOT}/checkpoints',
]

for d in DIRS:
    os.makedirs(d, exist_ok=True)

print('✅ Đã tạo cấu trúc thư mục:')
for d in DIRS:
    print(f'   📁 {d.replace(DRIVE_ROOT, "[Drive]/VietFood-Project")}')

In [ ]:
# ============================================================
# CELL 4: Tải VietFood67 từ Kaggle
# Dataset: 33,003 ảnh | 68 class | có bounding box (YOLO format)
# ============================================================
import subprocess

VIETFOOD67_DEST = f'{DRIVE_ROOT}/data/raw/VietFood67'

print('⬇️  Đang tải VietFood67 (~vài GB, mất khoảng 5-15 phút)...')
result = subprocess.run([
    'kaggle', 'datasets', 'download',
    '-d', 'thomasnguyen6868/vietfood68',
    '-p', VIETFOOD67_DEST,
    '--unzip'
], capture_output=True, text=True)

if result.returncode == 0:
    print('✅ VietFood67 đã tải và giải nén xong!')
else:
    print('❌ Lỗi khi tải VietFood67:')
    print(result.stderr)

In [ ]:
# ============================================================
# CELL 5: Tải 30VNFoods từ Kaggle
# Dataset: 25,136 ảnh | 30 class | classification (không có bbox)
# ============================================================
VNFOODS30_DEST = f'{DRIVE_ROOT}/data/raw/30VNFoods'

print('⬇️  Đang tải 30VNFoods (~vài GB, mất khoảng 5-10 phút)...')
result = subprocess.run([
    'kaggle', 'datasets', 'download',
    '-d', 'quandang/vietnamese-foods',
    '-p', VNFOODS30_DEST,
    '--unzip'
], capture_output=True, text=True)

if result.returncode == 0:
    print('✅ 30VNFoods đã tải và giải nén xong!')
else:
    print('❌ Lỗi khi tải 30VNFoods:')
    print(result.stderr)

In [ ]:
# ============================================================
# CELL 6: Kiểm tra cấu trúc thư mục sau khi tải
# ============================================================
import os

def inspect_dataset(root_path, dataset_name, max_depth=2):
    """In cây thư mục và đếm số file ảnh."""
    print(f'\n📂 {dataset_name}')
    print('=' * 50)

    if not os.path.exists(root_path):
        print(f'  ⚠️  Thư mục không tồn tại: {root_path}')
        return {}

    IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
    class_counts = {}
    total_images = 0
    total_files  = 0

    for root, dirs, files in os.walk(root_path):
        # Tính độ sâu tương đối
        depth = root.replace(root_path, '').count(os.sep)
        if depth > max_depth:
            continue

        indent = '  ' * depth
        folder_name = os.path.basename(root) or dataset_name

        imgs = [f for f in files if os.path.splitext(f)[1].lower() in IMAGE_EXTS]
        total_files += len(files)

        if depth > 0:  # Không in root
            print(f'{indent}📁 {folder_name}/ ({len(imgs)} ảnh)')

        if depth == max_depth and imgs:
            class_counts[folder_name] = len(imgs)
            total_images += len(imgs)

    print(f'\n📊 Tổng kết:')
    print(f'   Số class phát hiện : {len(class_counts)}')
    print(f'   Tổng số ảnh        : {total_images:,}')
    print(f'   Tổng số file       : {total_files:,}')
    return class_counts


counts_67  = inspect_dataset(f'{DRIVE_ROOT}/data/raw/VietFood67',  'VietFood67')
counts_30  = inspect_dataset(f'{DRIVE_ROOT}/data/raw/30VNFoods',   '30VNFoods')

In [ ]:
# ============================================================
# CELL 7: Tổng hợp báo cáo và lưu merge_report.txt
# ============================================================
import json
from datetime import datetime

# Gộp 2 dataset, cộng dồn số ảnh nếu trùng class
all_counts = {}
for cls, cnt in counts_67.items():
    all_counts[cls] = all_counts.get(cls, 0) + cnt
for cls, cnt in counts_30.items():
    all_counts[cls] = all_counts.get(cls, 0) + cnt

total_images = sum(all_counts.values())
total_classes = len(all_counts)

# Tìm class thiếu (dưới ngưỡng)
THRESHOLD = 100  # Ngưỡng tối thiểu theo task 1A
shortage = {cls: cnt for cls, cnt in all_counts.items() if cnt < THRESHOLD}

report_lines = [
    '=' * 60,
    f'MERGE REPORT — VietFood Project',
    f'Tạo lúc: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
    '=' * 60,
    f'Tổng số class  : {total_classes}',
    f'Tổng số ảnh    : {total_images:,}',
    f'Nguồn dữ liệu  : VietFood67 + 30VNFoods',
    '',
    f'Class có < {THRESHOLD} ảnh (cần bổ sung): {len(shortage)}',
    '-' * 40,
]
for cls, cnt in sorted(shortage.items(), key=lambda x: x[1]):
    report_lines.append(f'  {cls:<35} {cnt:>4} ảnh')

report_text = '\n'.join(report_lines)

# Lưu merge_report.txt
report_path = f'{DRIVE_ROOT}/reports/merge_report.txt'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(report_text)

print(report_text)
print(f'\n✅ Đã lưu báo cáo: {report_path}')

In [ ]:
# ============================================================
# CELL 8: Vẽ biểu đồ phân phối class & lưu shortage_classes.json
# (Task 2 của Nhóm 1A: EDA)
# ============================================================
import matplotlib.pyplot as plt
import matplotlib
import json

matplotlib.rcParams['font.family'] = 'DejaVu Sans'

if all_counts:
    sorted_counts = dict(sorted(all_counts.items(), key=lambda x: x[1], reverse=True))
    classes = list(sorted_counts.keys())
    counts  = list(sorted_counts.values())

    # --- Biểu đồ cột ---
    fig, ax = plt.subplots(figsize=(max(14, len(classes) * 0.3), 6))
    colors = ['#e74c3c' if c < THRESHOLD else '#2ecc71' for c in counts]
    bars = ax.bar(range(len(classes)), counts, color=colors, edgecolor='white', linewidth=0.5)

    ax.axhline(y=THRESHOLD, color='orange', linestyle='--', linewidth=1.5,
               label=f'Ngưỡng tối thiểu ({THRESHOLD} ảnh)')

    ax.set_xticks(range(len(classes)))
    ax.set_xticklabels(classes, rotation=90, fontsize=7)
    ax.set_ylabel('Số ảnh')
    ax.set_title('Phân phối số ảnh theo class — VietFood67 + 30VNFoods\n'
                 '🟥 Cần bổ sung  🟩 Đủ ngưỡng', fontsize=12)
    ax.legend()
    plt.tight_layout()

    chart_path = f'{DRIVE_ROOT}/reports/eda_distribution.png'
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Đã lưu biểu đồ: {chart_path}')

    # --- Lưu shortage_classes.json cho Nhóm 1B ---
    shortage_json = {
        'threshold': THRESHOLD,
        'generated_at': datetime.now().isoformat(),
        'shortage_classes': [
            {
                'class_name': cls,
                'current_count': cnt,
                'needed': max(0, 200 - cnt),  # Target 200 ảnh/class
                'priority': 'high' if cnt < 50 else 'medium'
            }
            for cls, cnt in sorted(shortage.items(), key=lambda x: x[1])
        ]
    }

    shortage_path = f'{DRIVE_ROOT}/reports/shortage_classes.json'
    with open(shortage_path, 'w', encoding='utf-8') as f:
        json.dump(shortage_json, f, ensure_ascii=False, indent=2)

    print(f'✅ Đã lưu shortage_classes.json: {shortage_path}')
    print(f'   → {len(shortage_json["shortage_classes"])} class cần bổ sung, gửi file này cho Nhóm 1B')
else:
    print('⚠️  Chưa có dữ liệu để vẽ biểu đồ. Hãy chạy lại Cell 6 sau khi tải dataset xong.')

In [ ]:
# ============================================================
# CELL 9: Kiểm tra file lỗi (ảnh không mở được)
# ============================================================
from PIL import Image
import os

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
broken_files = []
checked = 0

for dataset_path in [
    f'{DRIVE_ROOT}/data/raw/VietFood67',
    f'{DRIVE_ROOT}/data/raw/30VNFoods'
]:
    for root, _, files in os.walk(dataset_path):
        for fname in files:
            if os.path.splitext(fname)[1].lower() not in IMAGE_EXTS:
                continue
            fpath = os.path.join(root, fname)
            checked += 1
            try:
                with Image.open(fpath) as img:
                    img.verify()
            except Exception as e:
                broken_files.append({'path': fpath, 'error': str(e)})

            # In tiến độ mỗi 1000 ảnh
            if checked % 1000 == 0:
                print(f'  Đã kiểm tra {checked:,} ảnh, lỗi: {len(broken_files)}')

print(f'\n✅ Kiểm tra xong {checked:,} ảnh')
print(f'   File lỗi: {len(broken_files)}')

# Ghi danh sách file lỗi vào merge_report
if broken_files:
    broken_path = f'{DRIVE_ROOT}/reports/broken_files.json'
    with open(broken_path, 'w') as f:
        json.dump(broken_files, f, indent=2)
    print(f'   📄 Chi tiết lưu tại: {broken_path}')

In [ ]:
# ============================================================
# CELL 10: Tóm tắt kết quả — checklist bàn giao cho team
# ============================================================
print('=' * 60)
print('📋 CHECKLIST BÀN GIAO NHÓM 1A')
print('=' * 60)

checklist = [
    ('VietFood67 tải về Drive', os.path.exists(f'{DRIVE_ROOT}/data/raw/VietFood67')),
    ('30VNFoods tải về Drive',  os.path.exists(f'{DRIVE_ROOT}/data/raw/30VNFoods')),
    ('merge_report.txt',        os.path.exists(f'{DRIVE_ROOT}/reports/merge_report.txt')),
    ('eda_distribution.png',    os.path.exists(f'{DRIVE_ROOT}/reports/eda_distribution.png')),
    ('shortage_classes.json',   os.path.exists(f'{DRIVE_ROOT}/reports/shortage_classes.json')),
]

for name, done in checklist:
    status = '✅' if done else '❌'
    print(f'  {status}  {name}')

print()
print('📌 Bước tiếp theo:')
print('   → Gửi shortage_classes.json cho Nhóm 1B (trước 10h thứ 3)')
print('   → Nhóm 1B bắt đầu crawl ảnh bổ sung cho các class thiếu')
print('   → Nhóm 3 có thể dùng data/raw/ để test Dataset class')